# Capítulo 5 · Validación Cruzada

## Diplomado en Data Engineering

### Laboratorio interactivo: Hold-Out y K-Fold Cross Validation

En este cuaderno utilizaremos el dataset `salarios.csv` para estudiar cómo evaluar correctamente un modelo de Machine Learning.

El foco estará en responder:

- ¿Por qué una única partición Train/Test puede ser insuficiente?
- ¿Cómo funciona K-Fold Cross Validation?
- ¿Qué representan el promedio y la desviación estándar?
- ¿Qué ocurre después de validar un modelo?

## Objetivos

Al finalizar el laboratorio, el estudiante será capaz de:

- Implementar una evaluación Hold-Out.
- Analizar la variabilidad causada por distintas particiones.
- Aplicar K-Fold con K = 5 y K = 10.
- Interpretar los resultados por fold.
- Calcular promedio y desviación estándar.
- Comprender cómo se construye el modelo final para producción.

## 1. Importación de bibliotecas

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

from IPython.display import Markdown, display

from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_val_score
)

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

RANDOM_STATE = 42

## 2. Carga del dataset

In [2]:
df = pd.read_csv("salarios.csv")
df.head()

,AñosExperiencia,NivelEducacion,Puntaje,Salario
0,6.24,3,73.08,58436.10
1,14.31,4,78.02,100542.58
2,11.25,3,74.65,76953.98
3,9.38,1,63.31,62497.39
4,3.18,4,86.43,54024.34


## 3. Exploración inicial

El dataset contiene:

- `AñosExperiencia`
- `NivelEducacion`
- `Puntaje`
- `Salario`

In [3]:
print(f"Observaciones: {df.shape[0]}")
print(f"Variables: {df.shape[1]}")

display(df.describe().T)

Observaciones: 118
Variables: 4


,count,mean,std,min,25%,50%,75%,max
AñosExperiencia,118.0,7.043136,3.834155,0.45,4.005,7.060,10.1725,14.31
NivelEducacion,118.0,2.525424,1.122511,1.00,2.000,3.000,3.7500,4.00
Puntaje,118.0,73.341102,7.043034,60.64,67.465,73.225,78.9900,89.12
Salario,118.0,61325.851017,18038.895112,4800.00,48909.295,59352.130,73103.2750,100542.58


In [4]:
fig = px.scatter(
    df,
    x="AñosExperiencia",
    y="Salario",
    color="NivelEducacion",
    size="Puntaje",
    hover_data=["NivelEducacion", "Puntaje"],
    trendline="ols",
    title="Relación entre años de experiencia y salario"
)

fig.update_layout(
    height=560,
    xaxis_title="Años de experiencia",
    yaxis_title="Salario"
)

fig.show()

### Análisis breve

El salario tiende a aumentar con los años de experiencia. Sin embargo, también existen diferencias asociadas al nivel de educación, al puntaje y a observaciones atípicas.

Por esta razón, el resultado del modelo puede cambiar dependiendo de qué observaciones queden en entrenamiento y cuáles en prueba.

## 4. Preparación de variables

In [5]:
X = df[
    [
        "AñosExperiencia",
        "NivelEducacion",
        "Puntaje"
    ]
]

y = df["Salario"]

display(X.head())

,AñosExperiencia,NivelEducacion,Puntaje
0,6.24,3,73.08
1,14.31,4,78.02
2,11.25,3,74.65
3,9.38,1,63.31
4,3.18,4,86.43


# Parte I · Hold-Out

## 5. Una única partición Train/Test

Utilizaremos:

- 80% de los datos para entrenamiento.
- 20% para prueba.
- `random_state = 42`.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

modelo_holdout = LinearRegression()
modelo_holdout.fit(X_train, y_train)

pred_train = modelo_holdout.predict(X_train)
pred_test = modelo_holdout.predict(X_test)

r2_train = r2_score(y_train, pred_train)
r2_test = r2_score(y_test, pred_test)
mae_test = mean_absolute_error(y_test, pred_test)

print(f"R² entrenamiento: {r2_train:.3f}")
print(f"R² prueba: {r2_test:.3f}")
print(f"MAE prueba: {mae_test:,.0f}")

R² entrenamiento: 0.932
R² prueba: 0.957
MAE prueba: 2,767


In [7]:
resultado_holdout = pd.DataFrame({
    "Salario real": y_test,
    "Salario predicho": pred_test
})

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=resultado_holdout["Salario real"],
    y=resultado_holdout["Salario predicho"],
    mode="markers",
    name="Predicciones",
    hovertemplate=(
        "Real: %{x:,.0f}<br>"
        "Predicho: %{y:,.0f}"
        "<extra></extra>"
    )
))

limite_min = min(
    resultado_holdout["Salario real"].min(),
    resultado_holdout["Salario predicho"].min()
)

limite_max = max(
    resultado_holdout["Salario real"].max(),
    resultado_holdout["Salario predicho"].max()
)

fig.add_trace(go.Scatter(
    x=[limite_min, limite_max],
    y=[limite_min, limite_max],
    mode="lines",
    name="Predicción perfecta",
    line=dict(dash="dash")
))

fig.update_layout(
    title=f"Hold-Out · Valores reales vs. predichos · R² = {r2_test:.3f}",
    xaxis_title="Salario real",
    yaxis_title="Salario predicho",
    height=540
)

fig.show()

## 6. Variabilidad del Hold-Out

Repetiremos el mismo procedimiento usando distintas semillas.

El modelo, el dataset y la proporción Train/Test permanecen iguales. Solo cambia qué observaciones quedan en cada conjunto.

In [10]:
resultados_semillas = []

for semilla in range(1, 10):

    X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=semilla
    )

    modelo = LinearRegression()
    modelo.fit(X_train_s, y_train_s)

    pred_test_s = modelo.predict(X_test_s)

    resultados_semillas.append({
        "Semilla": semilla,
        "R² prueba": r2_score(y_test_s, pred_test_s),
        "MAE prueba": mean_absolute_error(y_test_s, pred_test_s)
    })

df_semillas = pd.DataFrame(resultados_semillas)

display(
    df_semillas.style.format({
        "R² prueba": "{:.3f}",
        "MAE prueba": "{:,.0f}"
    })
)

,Semilla,R² prueba,MAE prueba
0,1,0.980,"2,146"
1,2,0.961,"2,668"
2,3,0.973,"2,380"
3,4,0.973,"2,276"
4,5,0.979,"2,143"
5,6,0.979,"1,995"
6,7,0.864,"4,084"
7,8,0.982,"1,972"
8,9,0.973,"2,181"


In [9]:
fig = px.line(
    df_semillas,
    x="Semilla",
    y="R² prueba",
    markers=True,
    title="Variabilidad del R² según la partición Hold-Out"
)

fig.add_hline(
    y=df_semillas["R² prueba"].mean(),
    line_dash="dash",
    annotation_text=f"Promedio = {df_semillas['R² prueba'].mean():.3f}",
    annotation_position="top left"
)

fig.update_layout(
    height=520,
    xaxis_title="Random state",
    yaxis_title="R² de prueba"
)

fig.show()

print(f"R² mínimo: {df_semillas['R² prueba'].min():.3f}")
print(f"R² máximo: {df_semillas['R² prueba'].max():.3f}")
print(f"R² promedio: {df_semillas['R² prueba'].mean():.3f}")
print(f"Desviación estándar: {df_semillas['R² prueba'].std():.3f}")

R² mínimo: 0.842
R² máximo: 0.982
R² promedio: 0.965
Desviación estándar: 0.031


### Pregunta para discusión

El algoritmo y el dataset son los mismos, pero el R² cambia al modificar la semilla.

> ¿Cuál de todos estos resultados representa realmente la capacidad de generalización del modelo?



Conclusión: Una única partición Train/Test puede producir estimaciones optimistas o pesimistas del desempeño del modelo. Por esta razón, necesitamos un método que reduzca la dependencia de una sola división de los datos. Ese método es K-Fold Cross Validation.

# Parte II · K-Fold Cross Validation

## 7. Visualización del proceso K-Fold

In [11]:
print("K-Fold Cross Validation (K = 5)\n")

n_folds = 5

for fold in range(n_folds):

    linea = ""

    for i in range(n_folds):

        if i == fold:
            linea += "🟧 "
        else:
            linea += "🟦 "

    display(Markdown(f"**Fold {fold+1}**  \n{linea}"))

print("\n🟦 Entrenamiento")
print("🟧 Validación")

K-Fold Cross Validation (K = 5)



**Fold 1**  
🟧 🟦 🟦 🟦 🟦 

**Fold 2**  
🟦 🟧 🟦 🟦 🟦 

**Fold 3**  
🟦 🟦 🟧 🟦 🟦 

**Fold 4**  
🟦 🟦 🟦 🟧 🟦 

**Fold 5**  
🟦 🟦 🟦 🟦 🟧 


🟦 Entrenamiento
🟧 Validación


### Interpretación

- 🟦 Los bloques azules corresponden al conjunto de entrenamiento.
- 🟧 El bloque naranja corresponde al conjunto de validación.
- En cada iteración cambia el fold utilizado para validar.
- Al finalizar las cinco iteraciones, todas las observaciones han sido utilizadas exactamente una vez como conjunto de validación.

## 8. Aplicación de K-Fold con K = 5

In [12]:
modelo_cv = LinearRegression()

kfold_5 = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scores_k5 = cross_val_score(
    modelo_cv,
    X,
    y,
    cv=kfold_5,
    scoring="r2"
)

df_k5 = pd.DataFrame({
    "Fold": np.arange(1, 6),
    "R²": scores_k5
})

display(
    df_k5.style.format({
        "R²": "{:.3f}"
    })
)

print(f"R² promedio: {scores_k5.mean():.3f}")
print(f"Desviación estándar: {scores_k5.std():.3f}")

,Fold,R²
0,1,0.957
1,2,0.979
2,3,0.980
3,4,0.977
4,5,0.838


R² promedio: 0.946
Desviación estándar: 0.055


In [13]:
fig = px.bar(
    df_k5,
    x="Fold",
    y="R²",
    text="R²",
    title="R² obtenido en cada Fold · K = 5"
)

fig.update_traces(
    texttemplate="%{text:.3f}",
    textposition="outside"
)

fig.add_hline(
    y=scores_k5.mean(),
    line_dash="dash",
    annotation_text=f"Promedio = {scores_k5.mean():.3f}",
    annotation_position="top left"
)

fig.update_layout(
    height=520,
    yaxis_title="R² de validación"
)

fig.show()

### Interpretación de los resultados

- El promedio resume el desempeño esperado del modelo.
- La desviación estándar indica cuánto cambia el resultado entre folds.
- Un promedio alto y una desviación baja representan un modelo estable.
- Un promedio alto con desviación alta indica sensibilidad a la partición.

## 9. Comparación entre K = 5 y K = 10

In [14]:
kfold_10 = KFold(
    n_splits=10,
    shuffle=True,
    random_state=RANDOM_STATE
)

scores_k10 = cross_val_score(
    modelo_cv,
    X,
    y,
    cv=kfold_10,
    scoring="r2"
)

comparacion_k = pd.DataFrame({
    "Método": ["K-Fold K=5", "K-Fold K=10"],
    "R² promedio": [
        scores_k5.mean(),
        scores_k10.mean()
    ],
    "Desviación estándar": [
        scores_k5.std(),
        scores_k10.std()
    ],
    "Número de evaluaciones": [5, 10]
})

display(
    comparacion_k.style.format({
        "R² promedio": "{:.3f}",
        "Desviación estándar": "{:.3f}"
    })
)

,Método,R² promedio,Desviación estándar,Número de evaluaciones
0,K-Fold K=5,0.946,0.055,5
1,K-Fold K=10,0.950,0.064,10


# Parte III · Comparación final

## 10. Hold-Out vs. K-Fold

In [16]:
resumen_final = pd.DataFrame({
    "Método": [
        "Hold-Out, semilla 42",
        "Promedio de 30 Hold-Out",
        "K-Fold K=5",
        "K-Fold K=10"
    ],
    "R² estimado": [
        r2_test,
        df_semillas["R² prueba"].mean(),
        scores_k5.mean(),
        scores_k10.mean()
    ],
    "Desviación estándar": [
        np.nan,
        df_semillas["R² prueba"].std(),
        scores_k5.std(),
        scores_k10.std()
    ]
})

display(
    resumen_final.style.format({
        "R² estimado": "{:.3f}",
        "Desviación estándar": lambda valor: (
            "—" if pd.isna(valor) else f"{valor:.3f}"
        )
    })
)

,Método,R² estimado,Desviación estándar
0,"Hold-Out, semilla 42",0.957,—
1,Promedio de 30 Hold-Out,0.963,0.038
2,K-Fold K=5,0.946,0.055
3,K-Fold K=10,0.950,0.064


In [17]:
fig = px.bar(
    resumen_final,
    x="Método",
    y="R² estimado",
    error_y="Desviación estándar",
    text="R² estimado",
    title="Comparación de las estrategias de evaluación"
)

fig.update_traces(
    texttemplate="%{text:.3f}",
    textposition="outside"
)

fig.update_layout(
    height=540,
    yaxis_title="R² estimado"
)

fig.show()

# Parte IV · ¿Qué ocurre después de la validación cruzada?

La validación cruzada **no produce el modelo que será utilizado en producción**.

Su objetivo es estimar qué tan bien generaliza un algoritmo utilizando diferentes particiones del conjunto de datos.

Una vez que comprobamos que el modelo tiene un desempeño estable, se vuelve a entrenar utilizando el **100% de los datos disponibles**.

Ese nuevo modelo es el que finalmente se utiliza en producción.

## Flujo completo

```text
Dataset completo
       │
       ▼
K-Fold Cross Validation
       │
       ▼
Evaluación del desempeño
       │
       ▼
¿El modelo generaliza correctamente?
       │
      Sí
       │
       ▼
Entrenar nuevamente con el 100% de los datos
       │
       ▼
Modelo final para producción
```

## Diferencia entre validación y producción

| Validación Cruzada | Producción |
|---|---|
| Entrena varios modelos temporales | Entrena un único modelo final |
| Evalúa la capacidad de generalización | Construye el modelo definitivo |
| Utiliza distintas particiones | Utiliza el 100% del dataset |
| Los modelos no se despliegan | El modelo final se implementa |

## 11. Entrenamiento del modelo final

In [18]:
modelo_final = LinearRegression()

modelo_final.fit(X, y)

print("Modelo final entrenado con el 100% de los datos.")
print(f"Observaciones utilizadas: {len(X)}")

Modelo final entrenado con el 100% de los datos.
Observaciones utilizadas: 118


Una vez finalizada la validación cruzada ya sabemos que el algoritmo generaliza correctamente.

Ahora entrenaremos un único modelo utilizando el 100% de los datos disponibles.

Este será el modelo que se utilizaría en producción.

En esta etapa ya no tiene sentido calcular un R² de prueba, porque no existen datos reservados para evaluar el modelo.

### Idea clave

> Los modelos creados durante K-Fold sirven únicamente para evaluar.

> El modelo de producción se entrena nuevamente con todos los datos disponibles, una vez que sabemos que el algoritmo generaliza correctamente.

## 12. Conclusiones

- Hold-Out depende de una única partición.
- Cambiar la semilla puede modificar significativamente el resultado.
- K-Fold realiza múltiples evaluaciones.
- El promedio estima el desempeño esperado.
- La desviación estándar mide la estabilidad del modelo.
- Los modelos entrenados durante K-Fold no se llevan a producción.
- El modelo final se entrena nuevamente usando el 100% de los datos.

## Preguntas para discusión

1. ¿Por qué cambió el R² al modificar `random_state`?
2. ¿Qué representa el promedio de K-Fold?
3. ¿Qué representa la desviación estándar?
4. ¿Qué configuración fue más estable: K = 5 o K = 10?
5. ¿Por qué no se selecciona el modelo del mejor fold?
6. ¿Por qué el modelo final se entrena con el 100% de los datos?
7. ¿Cómo se relaciona K-Fold con GridSearchCV?

## Puente hacia el siguiente cuaderno

En el próximo laboratorio utilizaremos `GridSearchCV`.

Esta herramienta probará distintas combinaciones de hiperparámetros y aplicará K-Fold Cross Validation a cada combinación para seleccionar la configuración con mejor desempeño promedio.